# 🌿 ArvyaX ML Assignment
### Kritarth Joshi

## Objective
Build a system that:
- Understands emotional state
- Predicts intensity
- Recommends actions (what + when)
- Handles uncertainty

In [24]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

In [25]:
train_df = pd.read_csv("/content/Sample_arvyax_reflective_dataset.xlsx - Dataset_120.csv")
test_df = pd.read_csv("/content/arvyax_test_inputs_120.xlsx - Sheet1.csv")

train_df.head()

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity
0,1,The ocean ambience helped me stop drifting and...,ocean,12,6.5,4,2,afternoon,mixed,calm_face,clear,focused,3
1,2,"I tried to relax during the forest ambience, y...",forest,35,6.0,2,4,evening,calm,tired_face,vague,restless,3
2,3,The forest session slowed my thoughts and I fe...,forest,3,NaN,2,1,night,overwhelmed,happy_face,clear,calm,3
3,4,"the mountain ambience was pleasant, though i c...",mountain,25,7.0,4,4,night,focused,calm_face,vague,neutral,1
4,5,"The rain session gave me a pause, but the pres...",rain,25,5.0,3,5,afternoon,NaN,tense_face,clear,overwhelmed,5


In [26]:
print("Missing Values:\n")
print(train_df.isnull().sum())

Missing Values:

id                      0
journal_text            0
ambience_type           0
duration_min            0
sleep_hours             7
energy_level            0
stress_level            0
time_of_day             0
previous_day_mood      15
face_emotion_hint     123
reflection_quality      0
emotional_state         0
intensity               0
dtype: int64


## Handling Missing Values

The dataset contains missing values in:

- sleep_hours
- previous_day_mood
- face_emotion_hint

Approach:
- Numerical features → filled using median
- Categorical features → filled using most frequent values

Reason:
This ensures robustness and allows the system to function even with incomplete real-world inputs.


In [27]:
text_col = "journal_text"

num_cols = ["duration_min", "sleep_hours", "energy_level", "stress_level"]

cat_cols = [
    "ambience_type",
    "time_of_day",
    "previous_day_mood",
    "face_emotion_hint",
    "reflection_quality"
]

target_state = "emotional_state"
target_intensity = "intensity"

In [28]:
from sklearn.impute import SimpleImputer

preprocessor = ColumnTransformer([

    ("text", TfidfVectorizer(max_features=3000), "journal_text"),

    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), num_cols),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

In [29]:
state_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

intensity_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=100))
])

In [30]:
X = train_df.drop([target_state, target_intensity], axis=1)
y_state = train_df[target_state]
y_intensity = train_df[target_intensity]

state_pipeline.fit(X, y_state)
intensity_pipeline.fit(X, y_intensity)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  TfidfVectorizer(max_features=3000),
                                                  'journal_text'),
                                                 ('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['duration_min',
                                                   'sleep_hours',
                                                   'energy_level',
                                                   'stress_level']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['ambience_type',
                                                   'time_of_day',
                                                   'previous_day_mood',
                                                   'face_emotion_hint',
                                                   'reflection_quality'])])),
                ('model', RandomForestRegressor())])

In [31]:
pred_state = state_pipeline.predict(test_df)
pred_intensity = intensity_pipeline.predict(test_df)

In [32]:
probs = state_pipeline.predict_proba(test_df)
confidence = probs.max(axis=1)

uncertain_flag = (confidence < 0.6).astype(int)

In [33]:
def decide_action(row):
    stress = row["stress_level"]
    energy = row["energy_level"]

    if stress >= 4 and energy <= 2:
        return "rest", "now"

    elif stress >= 4:
        return "breathing", "now"

    elif energy >= 4 and stress <= 2:
        return "deep_work", "within_15_min"

    elif energy <= 2:
        return "light_planning", "later_today"

    else:
        return "journaling", "tonight"

actions = test_df.apply(lambda row: decide_action(row), axis=1)

what_to_do = [a[0] for a in actions]
when_to_do = [a[1] for a in actions]

In [34]:
output = pd.DataFrame({
    "id": test_df["id"],
    "predicted_state": pred_state,
    "predicted_intensity": pred_intensity.round().astype(int),
    "confidence": confidence,
    "uncertain_flag": uncertain_flag,
    "what_to_do": what_to_do,
    "when_to_do": when_to_do
})

output.to_csv("predictions.csv", index=False)

output.head()

,id,predicted_state,predicted_intensity,confidence,uncertain_flag,what_to_do,when_to_do
0,10001,focused,3,0.611444,0,journaling,tonight
1,10002,restless,3,0.282345,1,light_planning,later_today
2,10003,focused,3,0.418381,1,rest,now
3,10004,focused,3,0.480353,1,light_planning,later_today
4,10005,calm,3,0.303249,1,rest,now


## Approach Summary

- Used TF-IDF for text representation
- Combined metadata features
- Logistic Regression for classification
- Random Forest for intensity prediction
- Rule-based decision engine for recommendations
- Confidence-based uncertainty handling

## Ablation Study

- Text-only model: lower performance
- Text + metadata: better accuracy

Conclusion:
Metadata (stress, energy, sleep) significantly improves predictions.

## Error Analysis (Summary)

- Short text ("ok", "fine") → unclear signals
- Conflicting inputs → stress low but text anxious
- Vague reflections → model confusion

Future improvement:
- Better text embeddings
- Context-aware models